In [1]:

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
# ============================================
#  Função de normalização de strings e geração do dataset V3 - contendo apenas dados da prospecção
# ============================================


dataset = "v3"
df_treino = pd.read_csv(f"../datasets/df_treino_{dataset}.csv")
df_treino = df_treino.drop(columns= ["cavidade"])

y=df_treino["target"].copy()
X=df_treino.drop(columns=["target"]).copy()
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,

    test_size=0.2,
    random_state=42,
    stratify=y
)

df_treino.shape

(3957, 50)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib
from sasctl.pzmm import PickleModel
from sasctl.pzmm import ScoreCode
from sasctl.pzmm import JSONFiles
from sasctl.pzmm import ZipModel
from pathlib import Path

random_forest = RandomForestClassifier(
    n_estimators=1400,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

random_forest.fit(
    X_train,
    y_train
)

y_prob = random_forest.predict_proba(X_test)[:,1]


for threshold in [0.5, 0.45, 0.4, 0.35, 0.3]:

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    print(f"\nRandom forest - Dataset {dataset}\nThreshold: {threshold}")

    print(
        classification_report(
            y_test,
            y_pred
        )
    )

output_dir_model_pkl = Path("../artifacts")
output_dir_model_pkl.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(random_forest,  output_dir_model_pkl / f"random_forest_{dataset}.pkl")

output_dir_sasmodel = Path("../sas_model")
output_dir_sasmodel.mkdir(
    parents=True,
    exist_ok=True
)

PickleModel.pickle_trained_model(
    f"Espeleo-ML-RF-{dataset}",
    random_forest,
    output_dir_sasmodel
)


sc = ScoreCode()
sc.write_score_code(
    model_prefix= f"Espeleo-ML-RF-{dataset}",
    input_data=X_train,
    target_variable="target",
    predict_method=[random_forest.predict_proba,
                    [float, float]],
    target_index=1,
    target_values=[0,1],
    score_code_path=output_dir_sasmodel,
    model_file_name=f"Espeleo-ML-RF-{dataset}.pickle",
    predict_threshold=0.4
)




model_prefix = f"Espeleo-ML-RF-{dataset}"

model_path = output_dir_sasmodel / model_prefix
model_path.mkdir(parents=True, exist_ok=True)

JSONFiles.write_var_json(
    input_data=X_train,
    is_input=True,
    json_path=model_path
)


Random forest - Dataset v3
Threshold: 0.5
              precision    recall  f1-score   support

           0       0.84      0.97      0.90       617
           1       0.74      0.34      0.46       175

    accuracy                           0.83       792
   macro avg       0.79      0.65      0.68       792
weighted avg       0.82      0.83      0.80       792


Random forest - Dataset v3
Threshold: 0.45
              precision    recall  f1-score   support

           0       0.86      0.95      0.90       617
           1       0.70      0.43      0.54       175

    accuracy                           0.83       792
   macro avg       0.78      0.69      0.72       792
weighted avg       0.82      0.83      0.82       792


Random forest - Dataset v3
Threshold: 0.4
              precision    recall  f1-score   support

           0       0.87      0.92      0.89       617
           1       0.63      0.50      0.56       175

    accuracy                           0.83       79

c:\Users\sog3770\ML-Relev-ncia-de-Cavernas\.venv\lib\site-packages\sasctl\pzmm\write_score_code.py:2228: UserWarning: The model_prefix argument needs to be a valid Python function name. The provided value of Espeleo-ML-RF-v3 has been replaced with Espeleo_ML_RF_v3.
  warn(


FileNotFoundError: [Errno 2] No such file or directory: '..\\sas_model\\Espeleo_ML_RF_v3.pickle'